# 03 — Taxonomie & contrat ABSA

Définition de la taxonomie fermée et du format de sortie attendu.

In [2]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
while not (ROOT / "config" / "project_config.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "config" / "project_config.json").exists(), "Project root not found."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

CONFIG = json.loads((ROOT / "config" / "project_config.json").read_text(encoding="utf-8"))
SEED = CONFIG["random_seed"]
FIG_DIR = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)


ROOT: C:\Users\wiame.bourass\Downloads\sephora-consumer-voice-intelligence-llm-first\sephora-consumer-voice-intelligence-llm-first


## 1. Taxonomie

In [3]:
from src.absa_utils import load_taxonomy, taxonomy_table, core_aspect_ids, build_system_prompt
taxonomy = load_taxonomy(ROOT)
aspect_ids = core_aspect_ids(taxonomy)
tax_table = taxonomy_table(taxonomy)
display(tax_table)
print("Core aspects:", len(aspect_ids))

,aspect_id,display_name,description
0,efficacy_results,Efficacy / Results,Visible or perceived effectiveness and outcome...
1,hydration_dryness,Hydration / Dryness,"Moisture, hydration, dryness, dehydration or s..."
2,texture_finish,Texture / Finish,"Physical feel or finish: sticky, greasy, heavy..."
3,irritation_sensitivity,Irritation / Sensitivity,"Irritation, redness, burning, stinging, sensit..."
4,acne_breakouts,Acne / Breakouts,"Acne, pimples, breakouts, clogged pores, black..."
5,fragrance_smell,Fragrance / Smell,"Scent, fragrance, odor, smell or perfume perce..."
6,application_absorption,Application / Absorption,"Ease of application, spreading, absorption, la..."
7,packaging,Packaging,"Bottle, pump, jar, tube, cap, dispenser, leaka..."
8,price_value,Price / Value,"Price, affordability, expensive/cheap percepti..."


Core aspects: 9


## 2. Format de sortie

In [4]:
example = {
    "results": [
        {
            "segment_id": "example::s0",
            "aspects": [
                {"aspect_id": "hydration_dryness", "sentiment": "positive", "evidence": "keeps my skin hydrated"},
                {"aspect_id": "fragrance_smell", "sentiment": "negative", "evidence": "scent is too strong"}
            ]
        }
    ]
}
print(json.dumps(example, indent=2))

{
  "results": [
    {
      "segment_id": "example::s0",
      "aspects": [
        {
          "aspect_id": "hydration_dryness",
          "sentiment": "positive",
          "evidence": "keeps my skin hydrated"
        },
        {
          "aspect_id": "fragrance_smell",
          "sentiment": "negative",
          "evidence": "scent is too strong"
        }
      ]
    }
  ]
}


## 3. Prompt ABSA

Taxonomie fermée, sentiment unique par aspect et preuve extraite du texte.

In [5]:
prompt_version = CONFIG["llm"]["prompt_version"]
system_prompt = build_system_prompt(taxonomy, prompt_version)
print(system_prompt)
(ROOT / "prompts" / f"{prompt_version}_system_prompt.txt").write_text(system_prompt, encoding="utf-8")

You are an Aspect-Based Sentiment Analysis engine for skincare consumer reviews.
Prompt version: absa_v2_llm_first

Your task is extraction, not summarization.
For each input segment, identify zero, one, or several supported aspects from the CLOSED taxonomy below.
For each detected aspect assign exactly one sentiment: positive, neutral, or negative.
Return an evidence string copied EXACTLY from the input segment.
Do not infer an aspect from product name, brand, rating, skin type metadata, or general world knowledge.
Do not invent evidence. If no taxonomy aspect is explicitly supported by the segment, return an empty aspects list.
Do not return watchlist aspects or any label outside the closed taxonomy.

CLOSED TAXONOMY:
- efficacy_results: Visible or perceived effectiveness and outcome of the product.
- hydration_dryness: Moisture, hydration, dryness, dehydration or stripping sensation.
- texture_finish: Physical feel or finish: sticky, greasy, heavy, lightweight, silky, matte, shiny.


2352

## 4. Échantillon de segments

In [6]:
from src.sampling_utils import sample_segments
sample = sample_segments(ROOT, n=30, seed=SEED, min_words=CONFIG["min_words_for_llm"])
display(sample[[c for c in ["segment_id","product_name_catalog","skin_type","segment_text"] if c in sample.columns]].head(30))

,segment_id,product_name_catalog,skin_type,segment_text
0,reviews_0-250::255193::s0,C-Rush Vitamin C Gel Moisturizer,combination,10/10 recommend for dull skin or fine lines an...
1,reviews_750-1250::38940::s1,Melting Moisture Mask,combination,I only use it once a week (occasionally I’ll d...
2,reviews_750-1250::82280::s2,Multi-Peptide Eye Serum,combination,"I don’t remember to use it every single day, b..."
3,reviews_750-1250::1556::s5,Brightening Dark Circle Eye Cream,dry,It feels nourishing and moisturizingwithout be...
4,reviews_0-250::135903::s4,The True Cream Moisturizing Bomb,dry,I would definitely recommend this if you are s...
5,reviews_0-250::152420::s5,Take The Day Off Cleansing Balm Makeup Remover,combination,"I must admit, it’s a bit expensive, but I know..."
6,reviews_1250-end::43994::s1,Plumping High Performance Lip Filler with Hyal...,combination,It was sily and cooling and comforting.
7,reviews_500-750::87556::s1,Acne+ 2% BHA and Azelaic Acid Acne Spot Treatment,combination,"If that is the case for you as well, this may ..."
8,reviews_0-250::543022::s2,Vinopure Natural Salicylic Acid Pore Minimizin...,combination,"I have combination skin, so it did dry me out ..."
9,reviews_500-750::49720::s3,YUZU-C Beauty Sleeping Mask,combination,Definitely a night mask as it does leave some ...


## 5. Cas limites

- `Love it!` peut ne déclencher aucun aspect.
- Un segment peut contenir plusieurs aspects.
- `dry skin` décrit le profil et ne suffit pas à créer `hydration_dryness`.
- La preuve doit être présente dans le texte.

## Suite

Le notebook 04 teste le contrat sur 200 segments.